## CSV And Excel file -Structured data

In [1]:
import pandas as pd
import os


In [4]:
os.makedirs(r"data/structured_files", exist_ok=True)

In [5]:
# Create sample data
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Webcam'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics'],
    'Price': [999.99, 29.99, 79.99, 299.99, 89.99],
    'Stock': [50, 200, 150, 75, 100],
    'Description': [
        'High-performance laptop with 16GB RAM and 512GB SSD',
        'Wireless optical mouse with ergonomic design',
        'Mechanical keyboard with RGB backlighting',
        '27-inch 4K monitor with HDR support',
        '1080p webcam with noise cancellation'
    ]
}

# Save as CSV
df = pd.DataFrame(data)
df.to_csv('data/structured_files/products.csv', index=False)

In [6]:
# Save as Excel with multiple sheets
with pd.ExcelWriter('data/structured_files/inventory.xlsx') as writer:
    df.to_excel(writer, sheet_name='Products', index=False)
    
    # Add another sheet
    summary_data = {
        'Category': ['Electronics', 'Accessories'],
        'Total_Items': [3, 2],
        'Total_Value': [1389.97, 109.98]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)

## CSV processing

In [8]:
from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader

c:\Personal\upskill2025\Udemy_RAG_github\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
## Method 1 : Using CSVLoader : each row becomes a document
print("Using CSVLoader")
try:
    csv_loader=CSVLoader(
        file_path = "data/structured_files/products.csv",
        encoding ="utf-8",
        csv_args ={
            'delimiter' : ',',
            'quotechar' : '"',
        }
    )
    csv_docs=csv_loader.load()
    print(f" Loaded {len(csv_docs)} documents (one per row)")
    print("\n First Document : ")
    print(f" Content Review : {csv_docs[0].page_content}")
    print(f" Metadata : {csv_docs[0].metadata}")

except Exception as e:
    print(f"Error : {e}")

Using CSVLoader
 Loaded 5 documents (one per row)

 First Document : 
 Content Review : Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD
 Metadata : {'source': 'data/structured_files/products.csv', 'row': 0}


In [24]:
csv_docs[0]

Document(metadata={'source': 'data/structured_files/products.csv', 'row': 0}, page_content='Product: Laptop\nCategory: Electronics\nPrice: 999.99\nStock: 50\nDescription: High-performance laptop with 16GB RAM and 512GB SSD')

In [19]:
## Method 2 : Custom CSV processing for better control
print("Using Custom CSV processing")
from typing import List
from langchain_core.documents import Document

def process_csv_intelligently(filepath : str)-> List[Document]:
    df = pd.read_csv(filepath)
    documents=[]
    # Strategy 1 : one document per row with structured content
    for idx, row in df.iterrows():
        content = f"""Product Information : 
        Name: {row['Product']}
        Category: {row['Category']}
        Price: {row['Price']}
        Stock: {row['Stock']}
        Description: {row['Description']}"""

        # CReate document with rich metdata
        doc = Document(
            page_content=content,
            metadata = {
                'source' : filepath, 
                'row_index' : idx, 
                'product_name' : row['Product'], 
                'category' : row['Category'],
                'price' : row['Price'], 
                'data_type' : 'product_info'
            }
        )
        documents.append(doc)
    return documents

Using Custom CSV processing


In [ ]:
docc=process_csv_intelligently("data/structured_files/products.csv")
docc

In [16]:
docc[0]

Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'data_type': 'product_info'}, page_content='Product Information : \n        Name: Laptop\n        Category: Electronics\n        Price: 999.99\n        Stock: 50\n        Description: High-performance laptop with 16GB RAM and 512GB SSD')

In [25]:
# 📊 CSV Processing Strategies
print("\n📊 CSV Processing Strategies:")
print("\n1. Row-based (CSVLoader):")
print("  ✅ Simple one-row-one-document")
print("  ✅ Good for record lookups")
print("  ❌ Loses table context")

print("\n2. Intelligent Processing:")
print("  ✅ Preserves relationships")
print("  ✅ Creates summaries")
print("  ✅ Rich metadata")
print("  ✅ Better for Q&A")


📊 CSV Processing Strategies:

1. Row-based (CSVLoader):
  ✅ Simple one-row-one-document
  ✅ Good for record lookups
  ❌ Loses table context

2. Intelligent Processing:
  ✅ Preserves relationships
  ✅ Creates summaries
  ✅ Rich metadata
  ✅ Better for Q&A


## Excel Processing

In [27]:
## Method 1 : Using pandas: for full control
print("Using Pandas based excel processing")
def process_excel_with_pandas(filepath: str) -> List[Document]:
    documents = []

    excel_file =pd.ExcelFile(filepath)

    for sheet_name in excel_file.sheet_names:
        df = pd.read_excel(filepath, sheet_name = sheet_name)

        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {', '.join(df.columns)}\n"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content += df.to_string(index=False)
        
        doc = Document(
            page_content=sheet_content,
            metadata={
                'source': filepath,
                'sheet_name': sheet_name,
                'num_rows': len(df),
                'num_columns': len(df.columns),
                'data_type': 'excel_sheet'
            }
        )
        documents.append(doc)
    
    return documents




Using Pandas based excel processing


In [28]:
excel_docs = process_excel_with_pandas('data/structured_files/inventory.xlsx')
print(f"Processed {len(excel_docs)} sheets")

Processed 2 sheets


In [29]:
excel_docs

[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}, page_content='Sheet: Products\nColumns: Product, Category, Price, Stock, Description\nRows: 5\n\n Product    Category  Price  Stock                                         Description\n  Laptop Electronics 999.99     50 High-performance laptop with 16GB RAM and 512GB SSD\n   Mouse Accessories  29.99    200        Wireless optical mouse with ergonomic design\nKeyboard Accessories  79.99    150           Mechanical keyboard with RGB backlighting\n Monitor Electronics 299.99     75                 27-inch 4K monitor with HDR support\n  Webcam Electronics  89.99    100                1080p webcam with noise cancellation'),
 Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Summary', 'num_rows': 2, 'num_columns': 3, 'data_type': 'excel_sheet'}, page_content='Sheet: Summary\nColumns: Category, Total_Ite

In [42]:
## Method 2: Unstructured Excel Loader

from langchain_community.document_loaders import UnstructuredExcelLoader
# Method 2: UnstructuredExcelLoader
print("\n2️⃣ UnstructuredExcelLoader")
try:
    excel_loader = UnstructuredExcelLoader('data/structured_files/inventory.xlsx', mode="elements")
    unstructured_docs = excel_loader.load()
    print("Done")
except Exception as e:
    print("Error")


2️⃣ UnstructuredExcelLoader
Error


In [43]:
unstructured_docs

NameError: name 'unstructured_docs' is not defined

In [ ]:
print("  ✅ Handles complex Excel features")
print("  ✅ Preserves formatting info")
print("  ❌ Requires unstructured library")

print("  ℹ️ Requires unstructured library with Excel support")